# 🏛️ Notebook 08: Object-Oriented Programming (OOP) in Cloud Engineering
**Trainer: Sundeep Hiremat · TASK Nalgonda Center**  
**Course:** Cloud Software Developer (AWS DVA-C02 Aligned)

Welcome to the definitive guide on Object-Oriented Programming tailored for Cloud Developers!  
In real-world cloud SDKs (like AWS `boto3`, Kubernetes client, and Terraform CDKs), **everything is an object**.

### 🎯 What We Will Master in This Notebook:
1. **Classes vs Objects:** The architectural blueprint vs the running instance.
2. **Class Attributes vs Instance Attributes:** Global cloud config vs per-server metadata.
3. **Constructors (`__init__`) and `self`:** Lifecycle initialization.
4. **Encapsulation & Name Mangling:** Public, Protected (`_`), and Private (`__`) attributes with Python's name mangling mechanism (`_ClassName__attribute`).
5. **OOP Decorators:**
   * `@property` & `@name.setter`: Attribute validation & computed properties (e.g. validating IPv4 addresses and enforcing valid state transitions).
   * `@classmethod`: Alternative constructors (`from_json`, `from_template`).
   * `@staticmethod`: Pure utility helpers without instance state.
6. **Inheritance & Polymorphism:** Extending base cloud resources and overriding behavior.
7. **Abstract Base Classes (ABC):** Defining strict interfaces for multi-cloud drivers.
8. **Dunder / Magic Methods:** `__str__`, `__repr__`, `__eq__`, `__len__`.


---
## 1. The Basics: Classes, Objects & Attributes
A **Class** is the blueprint (like an AWS CloudFormation template or EC2 AMI).  
An **Object** is a specific running instance created from that blueprint (like `i-0987654321` running in `ap-south-1`).


In [ ]:
# 1. Simple Cloud Class
class BasicEC2Instance:
    # Class Attribute (Shared by all instances in the entire company)
    cloud_provider = "Amazon Web Services (AWS)"
    default_region = "ap-south-1"  # Mumbai

    def __init__(self, instance_id: str, instance_type: str = "t3.micro"):
        # Instance Attributes (Unique to each specific server)
        self.instance_id = instance_id
        self.instance_type = instance_type
        self.state = "STOPPED"
        print(f"✨ Created instance object {self.instance_id} [{self.instance_type}]")

    def start(self):
        self.state = "RUNNING"
        print(f"🟢 [{self.instance_id}] Server has booted and is now {self.state}.")

    def stop(self):
        self.state = "STOPPED"
        print(f"🔴 [{self.instance_id}] Server has shut down and is now {self.state}.")

# Creating instances (Instantiation)
server_web = BasicEC2Instance("i-web-001", instance_type="t3.medium")
server_db  = BasicEC2Instance("i-db-001", instance_type="r5.large")

# Testing class attributes vs instance attributes
print("\nCompany Cloud Provider:", BasicEC2Instance.cloud_provider)
print(f"Server Web: {server_web.instance_id} ({server_web.instance_type}) - State: {server_web.state}")
server_web.start()
print(f"Server DB:  {server_db.instance_id} ({server_db.instance_type}) - State: {server_db.state}")


---
## 2. Encapsulation, Privacy Levels & Name Mangling
In cloud computing, certain data must never be tampered with directly (such as IAM Secret Keys, CloudWatch root credentials, or internal lifecycle locks).

Python provides three levels of visibility:
1. **Public:** `self.name` (accessible anywhere).
2. **Protected:** `self._status` (convention: internal use only, subclasses can access).
3. **Private (with Name Mangling):** `self.__secret_key` (Python actively renames this attribute to prevent accidental overrides and outside tampering).

### 🔍 How Name Mangling Works:
When you name an attribute with two leading underscores `__variable`, Python internally transforms it into:
$$\_ClassName\_\_variable$$


In [ ]:
class SecureCloudServer:
    def __init__(self, instance_id: str, iam_secret_key: str):
        self.instance_id = instance_id          # Public
        self._operational_tier = "Tier-1"        # Protected (by convention)
        self.__iam_secret_key = iam_secret_key  # Private (Name Mangled!)
        self._is_encrypted = True

    def get_masked_key(self):
        """Safe method to view key without exposing plain text."""
        return f"****-****-{self.__iam_secret_key[-4:]}"

    def rotate_secret_key(self, new_key: str):
        if len(new_key) < 8:
            raise ValueError("Secret key too short! Must be at least 8 characters.")
        self.__iam_secret_key = new_key
        print(f"🔑 Secret key for {self.instance_id} rotated successfully.")

# Create a secure instance
secure_node = SecureCloudServer("i-sec-999", "AWS_SECRET_KEY_NALGONDA_2026")

# 1. Access Public & Protected
print("Public ID:", secure_node.instance_id)
print("Protected Tier:", secure_node._operational_tier)
print("Masked Key via method:", secure_node.get_masked_key())

# 2. Try accessing Private directly -> Raises AttributeError!
try:
    print(secure_node.__iam_secret_key)
except AttributeError as err:
    print("❌ DIRECT ACCESS BLOCKED:", err)

# 3. DEMONSTRATING NAME MANGLING:
# Notice how Python renamed '__iam_secret_key' to '_SecureCloudServer__iam_secret_key'
print("\n🔍 Inspecting object dictionary keys (dir):")
mangled_name = "_SecureCloudServer__iam_secret_key"
print(f"Direct access via mangled name ({mangled_name}):", getattr(secure_node, mangled_name))


---
## 3. OOP Decorators: `@property`, `@setter`, `@classmethod`, `@staticmethod`
Decorators inside classes provide clean, Pythonic ways to control data access and create specialized constructors.

### A. `@property` and `@setter`: Validation & State Protection
Suppose an EC2 instance can only transition through valid states: `PENDING` ➔ `RUNNING` ➔ `STOPPED` ➔ `TERMINATED`.  
Once terminated, it can **never** run again. We enforce this with a property setter!


In [ ]:
import re

class ManagedEC2Instance:
    VALID_STATES = {"PENDING", "RUNNING", "STOPPED", "TERMINATED"}

    def __init__(self, instance_id: str, ip_address: str):
        self.instance_id = instance_id
        self._state = "PENDING"
        self._ip_address = None
        self.ip_address = ip_address # triggers setter validation!

    # 1. Property Getter: State
    @property
    def state(self):
        """Read-only view of state (or custom formatted)."""
        return self._state

    # 2. Property Setter: State Transition Guard
    @state.setter
    def state(self, new_state: str):
        new_state = new_state.upper()
        if new_state not in self.VALID_STATES:
            raise ValueError(f"Invalid state: '{new_state}'. Allowed: {self.VALID_STATES}")
        if self._state == "TERMINATED":
            raise RuntimeError(f"Cannot change state of {self.instance_id}! Server is TERMINATED.")
        print(f"🔄 State Transition for {self.instance_id}: {self._state} ➔ {new_state}")
        self._state = new_state

    # 3. Property: IP Address with Regex Validation
    @property
    def ip_address(self):
        return self._ip_address

    @ip_address.setter
    def ip_address(self, ip: str):
        # Strict IPv4 regex validation
        pattern = r"^(\d{1,3}\.){3}\d{1,3}$"
        if not re.match(pattern, ip):
            raise ValueError(f"Invalid IPv4 Address format: '{ip}'")
        self._ip_address = ip

# Test validation
node = ManagedEC2Instance("i-prod-01", "13.233.45.10")
print(f"Node initialized: IP = {node.ip_address}, State = {node.state}")

# Transition state
node.state = "RUNNING"
node.state = "STOPPED"
node.state = "TERMINATED"

# Attempt invalid transition on terminated server -> raises error!
try:
    node.state = "RUNNING"
except RuntimeError as err:
    print("🛑 BLOCKED ILLEGAL TRANSITION:", err)

# Attempt invalid IP
try:
    node.ip_address = "999.invalid.ip.format"
except ValueError as err:
    print("🛑 BLOCKED INVALID IP:", err)


### B. `@classmethod`: Factory Constructors & Template Parsers
A **class method** takes `cls` (the class itself) instead of `self`.  
It is the standard pattern for building alternative constructors (e.g. creating an instance from a JSON string or AWS CLI template).


In [ ]:
import json

class CloudServerFactory:
    def __init__(self, instance_id: str, instance_type: str, region: str, cost_per_hr: float):
        self.instance_id = instance_id
        self.instance_type = instance_type
        self.region = region
        self.cost_per_hr = cost_per_hr

    # Standard constructor
    def __repr__(self):
        return f"<CloudServerFactory: {self.instance_id} [{self.instance_type}] in {self.region} (${self.cost_per_hr}/hr)>"

    # Alternative Constructor 1: From JSON string (e.g. from AWS API Gateway payload)
    @classmethod
    def from_json(cls, json_payload: str):
        data = json.loads(json_payload)
        return cls(
            instance_id=data["id"],
            instance_type=data.get("type", "t3.micro"),
            region=data.get("region", "ap-south-1"),
            cost_per_hr=data.get("cost", 0.0208)
        )

    # Alternative Constructor 2: Pre-configured Free Tier Template
    @classmethod
    def free_tier_template(cls, instance_id: str):
        return cls(
            instance_id=instance_id,
            instance_type="t2.micro",
            region="ap-south-1",
            cost_per_hr=0.0000 # Free tier eligible
        )

# 1. Normal creation
s1 = CloudServerFactory("i-001", "c5.xlarge", "us-east-1", 0.1700)

# 2. Created from API JSON string!
api_payload = '{"id": "i-json-777", "type": "t3.large", "region": "ap-south-1", "cost": 0.0832}'
s2 = CloudServerFactory.from_json(api_payload)

# 3. Created from Free Tier template!
s3 = CloudServerFactory.free_tier_template("i-free-999")

print("Created via normal init:      ", s1)
print("Created via from_json:        ", s2)
print("Created via free_tier_template:", s3)


### C. `@staticmethod`: Independent Utility Functions
A **static method** belongs to the class namespace but does not touch `self` or `cls`.  
It represents pure mathematical or validation utility functions.


In [ ]:
class CloudBillingCalculator:
    HOURLY_RATES = {
        "t2.micro": 0.0116,
        "t3.micro": 0.0208,
        "t3.medium": 0.0416,
        "c5.xlarge": 0.1700
    }

    # Static Utility Method 1: Sizing conversion
    @staticmethod
    def gib_to_tib(gib: float) -> float:
        """Converts GiB to TiB."""
        return round(gib / 1024, 4)

    # Static Utility Method 2: Projected Monthly Cost
    @staticmethod
    def estimate_monthly_cost(instance_type: str, hours_per_month: int = 730) -> float:
        rate = CloudBillingCalculator.HOURLY_RATES.get(instance_type)
        if not rate:
            raise ValueError(f"Unknown instance type: '{instance_type}'")
        return round(rate * hours_per_month, 2)

# Using static methods directly without creating any object!
print("10,240 GiB in TiB:", CloudBillingCalculator.gib_to_tib(10240))
print("Monthly cost for t3.micro (730 hrs):  $", CloudBillingCalculator.estimate_monthly_cost("t3.micro"))
print("Monthly cost for c5.xlarge (730 hrs): $", CloudBillingCalculator.estimate_monthly_cost("c5.xlarge"))


---
## 4. Inheritance, Polymorphism & Abstract Base Classes (ABC)
In cloud infrastructure management, we often handle diverse resources (EC2, S3, RDS, Lambda) through a **unified interface**.

### Abstract Base Class (Interface Contract)
Using Python's `abc` module, we define an abstract contract that guarantees every cloud resource must implement `.deploy()` and `.destroy()`.


In [ ]:
from abc import ABC, abstractmethod

# 1. Abstract Base Class
class AbstractCloudResource(ABC):
    def __init__(self, resource_id: str, region: str = "ap-south-1"):
        self.resource_id = resource_id
        self.region = region

    @abstractmethod
    def deploy(self):
        """Every child class MUST implement this method!"""
        pass

    @abstractmethod
    def destroy(self):
        """Every child class MUST implement this method!"""
        pass

# 2. Child Class: EC2 Compute
class EC2ComputeNode(AbstractCloudResource):
    def __init__(self, resource_id: str, instance_type: str = "t3.micro", region: str = "ap-south-1"):
        super().__init__(resource_id, region)
        self.instance_type = instance_type

    def deploy(self):
        print(f"🚀 [EC2] Launching {self.instance_type} virtual server ({self.resource_id}) in {self.region}...")

    def destroy(self):
        print(f"💥 [EC2] Terminating server ({self.resource_id}) and detaching root volume.")

# 3. Child Class: S3 Storage Bucket
class S3StorageBucket(AbstractCloudResource):
    def __init__(self, bucket_name: str, region: str = "ap-south-1"):
        super().__init__(bucket_name, region)

    def deploy(self):
        print(f"🪣 [S3] Creating S3 bucket 's3://{self.resource_id}' with private ACL in {self.region}...")

    def destroy(self):
        print(f"🗑️ [S3] Purging objects and deleting bucket 's3://{self.resource_id}'.")

# 4. POLYMORPHISM IN ACTION!
# We can loop through mixed cloud resources and manage them with a single loop:
infrastructure_stack = [
    EC2ComputeNode("i-web-alpha", instance_type="t3.medium"),
    EC2ComputeNode("i-web-beta",  instance_type="t3.medium"),
    S3StorageBucket("task-nalgonda-production-assets")
]

print("--- DEPLOYING ENTIRE CLOUD STACK ---")
for resource in infrastructure_stack:
    resource.deploy()

print("\n--- DESTROYING ENTIRE CLOUD STACK ---")
for resource in infrastructure_stack:
    resource.destroy()


---
## 5. Dunder / Magic Methods: `__str__`, `__repr__`, `__eq__`, `__len__`
Dunder (double underscore) methods allow our objects to integrate seamlessly with Python built-ins like `print()`, `==`, and `len()`.


In [ ]:
class EC2Cluster:
    def __init__(self, cluster_name: str):
        self.cluster_name = cluster_name
        self.nodes = []

    def add_node(self, node_id: str):
        self.nodes.append(node_id)

    # 1. len(cluster) -> returns count of nodes
    def __len__(self):
        return len(self.nodes)

    # 2. print(cluster) -> User-friendly string
    def __str__(self):
        return f"Cluster '{self.cluster_name}' [Total Active Nodes: {len(self.nodes)}]"

    # 3. repr(cluster) -> Unambiguous technical representation for developers/logs
    def __repr__(self):
        return f"EC2Cluster(cluster_name='{self.cluster_name}', nodes={self.nodes})"

    # 4. cluster1 == cluster2 -> Compare clusters by name
    def __eq__(self, other):
        if isinstance(other, EC2Cluster):
            return self.cluster_name == other.cluster_name
        return False

# Testing Magic Methods
cluster_hyd = EC2Cluster("ap-south-2-hyd-prod")
cluster_hyd.add_node("i-001")
cluster_hyd.add_node("i-002")
cluster_hyd.add_node("i-003")

print("len():  ", len(cluster_hyd))           # 3
print("str():  ", str(cluster_hyd))           # User-friendly
print("repr(): ", repr(cluster_hyd))          # Developer debug view

cluster_clone = EC2Cluster("ap-south-2-hyd-prod")
print("Are clusters equal (__eq__)?", cluster_hyd == cluster_clone) # True
